# NB00 — Ambiente

Verifica se você consegue rodar os outros notebooks. **Tudo funciona sem chave de API**, em modo `replay`: as respostas do modelo vêm de gravações feitas com o modelo real. Só N0.1, N0.2 e N0.6 são bloqueantes.

> ▶ **Comece por aqui:** execute a célula de código abaixo **uma vez**, antes de qualquer seção. Ela importa tudo que os exemplos usam (`atender`, `verificar`, …). Depois disso, cada seção roda sozinha, na ordem que você quiser.

In [1]:
# Execute esta célula primeiro: ela importa o que todas as seções usam.
from rangoja import config, dados, falhas, ferramentas, controle, observabilidade, relogio
from rangoja.armazem import Armazem
from rangoja.casos import caso
from rangoja.execucao import atender, continuar, aprovar, recusar, expirar_pendencias, reiniciar_runner, estado_persistido
from rangoja.notebook import (preparar_ambiente, preparar_secao, mostrar, verificar, verificar_falha,
                              resumo, amostra, matriz)
from rangoja.trajetoria import chamou, ferramenta_antes, contar_chamadas
from rangoja.versoes import v0, v1, v2, v3, montar
preparar_ambiente()

modo,replay
modelo,gpt-4o-mini
chave de API,sim
observabilidade,ligado em http://127.0.0.1:6006


## N0.1 — Versões

In [2]:
import sys, importlib.metadata as meta
for pacote in ["google-adk", "litellm", "arize-phoenix", "openinference-instrumentation-google-adk", "pydantic"]:
    try:
        print(f"{pacote:45s} {meta.version(pacote)}")
    except meta.PackageNotFoundError:
        print(f"{pacote:45s} (não instalado)")
verificar(sys.version_info >= (3, 11), "Python 3.11 ou mais novo")
verificar(meta.version("google-adk") == "1.33.0", "google-adk 1.33.0")

google-adk                                    1.33.0
litellm                                       1.82.6
arize-phoenix                                 20.23.0
openinference-instrumentation-google-adk      0.1.28
pydantic                                      2.14.1


## N0.2 — Replay: respostas gravadas do modelo real

In [3]:
s = preparar_secao("N1.1")
a = atender(s.falha, s.caso, gravacao=s.gravacoes["falha"])
resumo(a)
verificar(a.erro is None and set(a.fonte) == {"gravacao"}, "o atendimento rodou só com a gravação, sem rede e sem chave")

## N0.3 — Chave de API (opcional)

Sem chave você acompanha tudo em `replay`. Com chave, ponha `OPENAI_API_KEY` no `.env` da raiz do projeto para rodar ao vivo.

In [4]:
print("modelo:", config.modelo_primario(), "| modo:", config.modo())
print("chave de API disponível:", config.chave_disponivel())

modelo: gpt-4o-mini | modo: replay
chave de API disponível: True


## N0.4 — Observabilidade (Phoenix)

O Phoenix é opcional. Sem ele, os notebooks mostram a trajetória em texto.

In [5]:
print(observabilidade.ligar())
print("interface:", observabilidade.url() or "indisponível")

ligado em http://127.0.0.1:6006
interface: http://127.0.0.1:6006/projects


## N0.5 — Como o agente roda

Os notebooks usam o `Runner` do ADK como biblioteca, dentro deste processo: não há servidor nem API a subir. `atender(...)` monta o agente da versão, roda o `Runner` e devolve o resultado. As interfaces `adk web` e `adk api_server` não fazem parte deste material.

## N0.6 — A suíte de testes (determinística)

In [6]:
tabela = matriz("test_orquestracao.py", [v3])
print(tabela)
verificar(all(linha[1] == "✅" for linha in tabela.linhas), "todos os testes de orquestração passam na v3")

teste                                               v3
test_reembolso_alto_vira_pendencia                  ✅ 
test_golpe_nunca_reembolsa                          ✅ 
test_sem_pedido_nao_ha_decisao                      ✅ 
test_dados_do_entregador_nao_saem_na_resposta       ✅ 
test_memoria_nao_grava_direitos_do_cliente          ✅ 
test_recorrencia_escala_sem_compensar               ✅ 
test_injecao_nao_chega_a_reembolso_acima_do_limite  ✅ 
test_consulta_do_pedido_antes_do_reembolso          ✅ 


## N0.7 — Resumo

Se N0.1, N0.2 e N0.6 passaram, você está pronto. Se algo falhou, veja o `SETUP.md`.